In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import os

In [2]:
# Define the CNN architecture
class FacialKeypointsCNN(nn.Module):
    def __init__(self):
        super(FacialKeypointsCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2, 2) # 48x48
        
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2, 2) # 24x24
        
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.relu3 = nn.ReLU()
        self.pool3 = nn.MaxPool2d(2, 2) # 12x12
        
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.relu4 = nn.ReLU()
        self.pool4 = nn.MaxPool2d(2, 2) # 6x6
        
        self.fc1 = nn.Linear(256 * 6 * 6, 512)
        self.relu5 = nn.ReLU()
        self.fc2 = nn.Linear(512, 30) # 15 keypoints * 2 (x, y)

    def forward(self, x):
        x = self.pool1(self.relu1(self.conv1(x)))
        x = self.pool2(self.relu2(self.conv2(x)))
        x = self.pool3(self.relu3(self.conv3(x)))
        x = self.pool4(self.relu4(self.conv4(x)))
        x = x.view(-1, 256 * 6 * 6)
        x = self.relu5(self.fc1(x))
        x = self.fc2(x)
        return x

In [3]:
class FacialKeypointsDataset(Dataset):
    def __init__(self, csv_file, train=True):
        self.train = train
        self.data = pd.read_csv(csv_file)
        
        if self.train:
            # Drop rows with missing keypoints for simplicity
            self.data = self.data.dropna()
            self.keypoints = self.data.iloc[:, :-1].values.astype(np.float32)
        
        self.images = self.data['Image'].values

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image_str = self.images[idx]
        image = np.array([int(p) for p in image_str.split()]).reshape(96, 96).astype(np.float32)
        # Normalize image to [0, 1]
        image = image / 255.0
        # Add channel dimension
        image = np.expand_dims(image, axis=0)
        
        if self.train:
            keypoints = self.keypoints[idx]
            # Normalize keypoints to [-1, 1]
            keypoints = (keypoints - 48.0) / 48.0
            return torch.tensor(image), torch.tensor(keypoints)
        else:
            return torch.tensor(image)

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Hyperparameters
batch_size = 64
epochs = 15
learning_rate = 0.001

print("Loading data...")
train_dataset = FacialKeypointsDataset('training.csv', train=True)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

model = FacialKeypointsCNN().to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

Using device: cpu
Loading data...


In [5]:
print("Starting training...")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for i, (images, keypoints) in enumerate(train_loader):
        images = images.to(device)
        keypoints = keypoints.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, keypoints)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

print("Training finished. Saving model...")
torch.save(model.state_dict(), 'model.pth')

Starting training...
Epoch [1/15], Loss: 0.0762
Epoch [2/15], Loss: 0.0052
Epoch [3/15], Loss: 0.0045
Epoch [4/15], Loss: 0.0045
Epoch [5/15], Loss: 0.0044
Epoch [6/15], Loss: 0.0044
Epoch [7/15], Loss: 0.0042
Epoch [8/15], Loss: 0.0038
Epoch [9/15], Loss: 0.0034
Epoch [10/15], Loss: 0.0029
Epoch [11/15], Loss: 0.0024
Epoch [12/15], Loss: 0.0021
Epoch [13/15], Loss: 0.0019
Epoch [14/15], Loss: 0.0017
Epoch [15/15], Loss: 0.0016
Training finished. Saving model...


In [6]:
print("Generating predictions on test set...")
test_dataset = FacialKeypointsDataset('test.csv', train=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

model.eval()
all_predictions = []

with torch.no_grad():
    for images in test_loader:
        images = images.to(device)
        outputs = model(images)
        outputs = outputs.cpu().numpy()
        
        # Unnormalize keypoints: [-1, 1] -> [0, 96]
        outputs = (outputs * 48.0) + 48.0
        all_predictions.extend(outputs)

columns = [
    "left_eye_center_x","left_eye_center_y","right_eye_center_x","right_eye_center_y",
    "left_eye_inner_corner_x","left_eye_inner_corner_y","left_eye_outer_corner_x","left_eye_outer_corner_y",
    "right_eye_inner_corner_x","right_eye_inner_corner_y","right_eye_outer_corner_x","right_eye_outer_corner_y",
    "left_eyebrow_inner_end_x","left_eyebrow_inner_end_y","left_eyebrow_outer_end_x","left_eyebrow_outer_end_y",
    "right_eyebrow_inner_end_x","right_eyebrow_inner_end_y","right_eyebrow_outer_end_x","right_eyebrow_outer_end_y",
    "nose_tip_x","nose_tip_y","mouth_left_corner_x","mouth_left_corner_y","mouth_right_corner_x","mouth_right_corner_y",
    "mouth_center_top_lip_x","mouth_center_top_lip_y","mouth_center_bottom_lip_x","mouth_center_bottom_lip_y"
]

# Create wide DataFrame with ImageId
submission_df = pd.DataFrame(all_predictions, columns=columns)
submission_df.insert(0, 'ImageId', range(1, len(submission_df) + 1))

# Map to Kaggle submission format using IdLookupTable.csv
try:
    lookup_df = pd.read_csv('IdLookupTable.csv')
except FileNotFoundError:
    try:
        lookup_df = pd.read_csv('submissionFileFormat.csv')
    except FileNotFoundError:
        print("Error: Could not find IdLookupTable.csv or submissionFileFormat.csv")
        lookup_df = None

if lookup_df is not None:
    # Melt the wide DataFrame into a long format: ImageId | FeatureName | Location
    long_df = pd.melt(submission_df, id_vars=['ImageId'], var_name='FeatureName', value_name='Location')
    
    # The lookup table might already have a 'Location' column (mostly empty), we drop it to avoid _x, _y suffixes during merge
    if 'Location' in lookup_df.columns:
        lookup_df = lookup_df.drop(columns=['Location'])
    
    # Merge with the lookup table on ImageId and FeatureName to get RowId
    merged_df = pd.merge(lookup_df, long_df, on=['ImageId', 'FeatureName'], how='left')
    
    # Some values could be outside the [0, 96] range due to model inaccuracies. Clip them.
    merged_df['Location'] = merged_df['Location'].clip(0, 96)
    
    # Final format: RowId, Location
    final_submission = merged_df[['RowId', 'Location']]
    
    # Save the final submission file
    final_submission.to_csv('submission.csv', index=False)
    print("Predictions saved to submission.csv in Kaggle format!")
else:
    # Fallback to saving wide format
    submission_df.to_csv('predictions.csv', index=False)
    print("Saved predictions.csv (wide format).")


Generating predictions on test set...
Predictions saved to submission.csv in Kaggle format!
